# 01 · Prepare and review flight profiles

Run cells from top to bottom with **Shift+Enter**, using the **Python (HERA)**
kernel. This notebook prepares measurements. Notebook 02 trains the models.

The original recordings stay unchanged. Every preparation run creates new
output folders. Phase boundaries, exposure and pressure accuracy are still
provisional; a successful run does not verify those assumptions.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Image

# Works when Jupyter starts in the repository or its notebooks folder.
here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents)
             if (p / "src/prepare_profiles.py").is_file()
             and (p / "src/fit_profile.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside HERA-Machine-Learning/notebooks.")
sys.path.insert(0, str(ROOT / "src"))
print("Project:", ROOT)
print("Python:", sys.executable)

## Choose the flight and detectors

Start with these settings. For the bin-size check later, change only
`BIN_SECONDS` to 20 or 30. For another date, first create and review its own
configuration files; never copy June 1's phase times without checking them.

In [ ]:
FLIGHT_ID = "2025-06-01"
DETECTORS = ["top", "bottom"]
BIN_SECONDS = 10

## Check the input files

Each configuration identifies a detector recording and logger file. The table
below shows the exact inputs before anything is processed. Temperature, pulse
amplitudes and other raw measurements remain in the original files.

In [ ]:
import math

if not math.isfinite(BIN_SECONDS) or BIN_SECONDS <= 0:
    raise ValueError("BIN_SECONDS must be positive and finite.")
if not DETECTORS or len(DETECTORS) != len(set(DETECTORS)):
    raise ValueError("Choose at least one detector, without duplicates.")

configurations = {}
inventory = []
for detector in DETECTORS:
    config_path = ROOT / "configs" / f"{FLIGHT_ID}_{detector}.json"
    cfg = json.loads(config_path.read_text(encoding="utf-8"))
    if (cfg["flight_id"], cfg["detector_id"]) != (FLIGHT_ID, detector):
        raise ValueError(f"Configuration identity mismatch: {config_path}")
    cfg["bin_seconds"] = float(BIN_SECONDS)
    configurations[detector] = cfg
    for key in ("detector_file", "logger_file"):
        source = ROOT / cfg[key]
        if not source.is_file():
            raise FileNotFoundError(f"Missing input: {source}")
        inventory.append({"detector": detector, "role": key,
                          "file": str(source), "size_MiB": round(source.stat().st_size / 2**20, 2)})
display(pd.DataFrame(inventory))

## Run preparation

This cell checks recording integrity, counts events in time bins, flags pressure
spikes and calculates provisional exposure. It uses `src/prepare_profiles.py`.

The temporary configuration changes only the requested bin width. Your original
configuration files remain unchanged; the full configuration used is preserved
in each preparation report.

In [ ]:
import tempfile
import prepare_profiles

prepared_runs = {}
with tempfile.TemporaryDirectory(prefix="hera_prepare_") as temporary:
    for detector, cfg in configurations.items():
        run_config = Path(temporary) / f"{detector}.json"
        run_config.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
        report, result_dir, clean_dir = prepare_profiles.prepare(ROOT, run_config)
        csv_path = clean_dir / f"{detector}_bins_{float(BIN_SECONDS):g}s.csv"
        if not csv_path.is_file():
            raise FileNotFoundError("Use the updated prepare_profiles.py included with these notebooks.")
        prepared_runs[detector] = {"report": report, "results": result_dir,
                                   "csv": csv_path}

## Review counts and quality flags

The accounting check includes events in partial recording-edge intervals.
Flagging a pressure reading does not remove its detector event from this table.
The training script makes its own documented selection of bins.

In [ ]:
rows = []
for detector, run in prepared_runs.items():
    report = run["report"]
    accounted = report["binned_events"] + report["edge_events_before"] + report["edge_events_after"]
    assert accounted == report["source_events"], "Event accounting failed."
    rows.append({"detector": detector, "source_events": report["source_events"],
                 "binned_events": report["binned_events"],
                 "edge_events": report["edge_events_before"] + report["edge_events_after"],
                 "bins": report["bins"],
                 "pressure_readings_flagged": report["pressure_flagged_event_rows"],
                 "all_events_accounted_for": accounted == report["source_events"]})
display(pd.DataFrame(rows))

In [ ]:
for detector, run in prepared_runs.items():
    print(detector.upper(), "— measured profile and clock review")
    display(Image(filename=str(run["results"] / f"{detector}_profile_review.png"), width=1000))

## Inspect the prepared table

Ground periods remain in this table for review. Notebook 02 selects ascent and
descent. `training_ready=False` is preserved because scientific assumptions are
unresolved; exploratory model fitting is still allowed.

In [ ]:
for detector, run in prepared_runs.items():
    bins = pd.read_csv(run["csv"])
    print(detector.upper(), "—", run["csv"])
    display(bins.groupby("phase_provisional").agg(
        bins=("counts_all", "size"), events=("counts_all", "sum"),
        pressure_min_pa=("pressure_pa", "min"), pressure_max_pa=("pressure_pa", "max")))
    display(bins[["bin_start_s", "phase_provisional", "counts_all", "elapsed_s",
                  "live_time_s_candidate", "pressure_pa", "pressure_flags",
                  "training_ready"]].head(6))

In [ ]:
for detector, run in prepared_runs.items():
    flags = pd.read_csv(run["results"] / "pressure_flags.csv")
    print(f"{detector}: {len(flags)} flagged pressure readings")
    if not flags.empty:
        display(flags[["source_record", "time_s", "pressure_pa"]].head(10))

## Next: train in notebook 02

Open **02_train_and_compare.ipynb** and use the same flight, detectors and bin
width. It finds the latest matching prepared tables and prints the selected
paths before training. You can pin those paths for an exact rerun.

Do not edit the prepared CSV to remove inconvenient observations. Change a
documented processing choice and rerun preparation when needed.